# Support Vector Machines (SVM)

## SVM

Implementing a Support Vector Machine (SVM) classifier from scratch using Stochastic Gradient Descent (SGD) algorithm for training.

* Implementing the `SVM` class below. It holds logic for an SVM classifier using the Hinge loss.
* Testing it using synthetic data first (i.e. `make_blobs` and `make_moons` for linearly separable and non-separable tasks).
* Finally, running the final SVM implementation of the [Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine) dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class SVM:
    """
    Linear SVM with hinge loss and L2 regularization.
    Trained using simple SGD over individual samples.
    """

    def __init__(self, learning_rate=0.01, lambda_reg=0.01, epochs=30, shuffle=True, random_state=0):
        self.learning_rate = float(learning_rate)
        self.lambda_reg = float(lambda_reg)
        self.epochs = int(epochs)
        self.shuffle = bool(shuffle)
        self.random_state = int(random_state)
        self.w = None
        self.b = 0.0
        self.classes_ = None
        self._rs = np.random.RandomState(self.random_state)

    def _to_pm1(self, y):
        # If already -1/+1, keep it. Otherwise map two unique classes -> {-1, +1}.
        uniq = np.unique(y)
        if uniq.size != 2:
            raise ValueError("SVM expects binary labels. Wrap with one-vs-rest for multi-class.")
        if np.array_equal(np.sort(uniq), np.array([-1, 1])):
            return y.astype(float), np.array([-1, 1])
        y_bin = np.where(y == uniq[0], -1.0, 1.0)
        return y_bin, uniq

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        y_bin, classes = self._to_pm1(y)
        n, d = X.shape
        self.w = np.zeros(d, dtype=float)
        self.b = 0.0
        self.classes_ = classes  # to map predictions back

        for _ in range(self.epochs):
            idx = np.arange(n)
            if self.shuffle:
                self._rs.shuffle(idx)
            for i in idx:
                xi = X[i]
                yi = y_bin[i]
                margin = yi * (np.dot(self.w, xi) + self.b)
                if margin >= 1.0:
                    # only regularization on w
                    self.w -= self.learning_rate * (self.lambda_reg * self.w)
                else:
                    # regularization + hinge gradient
                    self.w -= self.learning_rate * (self.lambda_reg * self.w - yi * xi)
                    self.b += self.learning_rate * yi
        return self

    def decision_function(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        y_pm1 = np.sign(scores)
        y_pm1[y_pm1 == 0] = 1.0
        # Map back to original labels if needed
        if np.array_equal(np.sort(self.classes_), np.array([-1, 1])):
            return y_pm1.astype(int)
        return np.where(y_pm1 == -1.0, self.classes_[0], self.classes_[1])


In [ ]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_wine

In [ ]:
# Blobs (mostly linear)
Xb, yb = make_blobs(n_samples=600, centers=2, cluster_std=2.0, random_state=7)
Xb = (Xb - Xb.mean(axis=0)) / (Xb.std(axis=0) + 1e-9)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=7, stratify=yb)

lin_svm = SVM(learning_rate=0.01, lambda_reg=0.01, epochs=30, random_state=7)
lin_svm.fit(Xb_tr, yb_tr)
print("Blobs accuracy (linear):", (lin_svm.predict(Xb_te) == yb_te).mean())

# Moons (non-linear)
Xm, ym = make_moons(n_samples=600, noise=0.25, random_state=13)
Xm = (Xm - Xm.mean(axis=0)) / (Xm.std(axis=0) + 1e-9)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=13, stratify=ym)

lin_svm2 = SVM(learning_rate=0.01, lambda_reg=0.01, epochs=30, random_state=13)
lin_svm2.fit(Xm_tr, ym_tr)
print("Moons accuracy (linear):", (lin_svm2.predict(Xm_te) == ym_te).mean())


Blobs accuracy (linear): 0.9833333333333333
Moons accuracy (linear): 0.8444444444444444


In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine()
X, y = wine.data.astype(float), wine.target.astype(int)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-9)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

def fit_ovr_svm(X_tr, y_tr, **svm_params):
    classes = np.unique(y_tr)
    models = []
    for c in classes:
        y_bin = np.where(y_tr == c, 1, -1)
        m = SVM(**svm_params)
        m.fit(X_tr, y_bin)
        models.append((c, m))
    return models

def predict_ovr(models, X):
    # choose class with largest decision score
    scores = np.column_stack([m.decision_function(X) for (_, m) in models])
    best = np.argmax(scores, axis=1)
    classes = np.array([c for (c, _) in models])
    return classes[best]

ovr_linear = fit_ovr_svm(X_tr, y_tr, learning_rate=0.01, lambda_reg=0.01, epochs=60, random_state=0)
y_hat_lin = predict_ovr(ovr_linear, X_te)
acc_lin = (y_hat_lin == y_te).mean()
print(f"Wine accuracy (linear SVM, OVR): {acc_lin:.3f}")


Wine accuracy (linear SVM, OVR): 1.000


## SVM with Kernel Trick

* Extending the SVM implementation to support the kernel trick with the Radial Basis Function (RBF) kernel. Performance reporting on the [Wine Dataset](https://archive.ics.uci.edu/dataset/109/wine) dataset and comparing to the linear SVM.

In [ ]:
class KernelSVM:
    """
    RBF kernel SVM in the dual with simple projected gradient updates.
    """

    def __init__(self, C=1.0, gamma=0.5, epochs=120, learning_rate=0.01, random_state=0):
        self.C = float(C)
        self.gamma = float(gamma)
        self.epochs = int(epochs)
        self.learning_rate = float(learning_rate)
        self.random_state = int(random_state)
        self._rs = np.random.RandomState(self.random_state)

        self.X_train = None
        self.y_train = None
        self.alpha = None
        self.b = 0.0
        self.classes_ = None
        self._K = None

    def kernel_function(self, X1, X2):
        X1 = np.asarray(X1, dtype=float)
        X2 = np.asarray(X2, dtype=float)
        # RBF: exp(-gamma * ||x - z||^2)
        X1_sq = np.sum(X1**2, axis=1, keepdims=True)
        X2_sq = np.sum(X2**2, axis=1, keepdims=True).T
        sq_dists = X1_sq + X2_sq - 2.0 * (X1 @ X2.T)
        return np.exp(-self.gamma * sq_dists)

    def kernelfunction(self, X1, X2):
        return self.kernel_function(X1, X2)

    def _to_pm1(self, y):
        uniq = np.unique(y)
        if uniq.size != 2:
            raise ValueError("KernelSVM expects binary labels. Use OVR for multi-class.")
        if np.array_equal(np.sort(uniq), np.array([-1, 1])):
            return y.astype(float), np.array([-1, 1])
        y_bin = np.where(y == uniq[0], -1.0, 1.0)
        return y_bin, uniq

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        y_bin, classes = self._to_pm1(y)
        n = X.shape[0]

        self.X_train = X
        self.y_train = y_bin
        self.classes_ = classes
        self.alpha = np.zeros(n, dtype=float)
        self.b = 0.0
        self._K = self.kernel_function(X, X)

        # Maximize dual: sum alpha_i - 0.5 sum_ij alpha_i alpha_j y_i y_j K_ij
        for _ in range(self.epochs):
            idx = self._rs.permutation(n)
            for i in idx:
                # gradient wrt alpha_i
                f_i = np.dot(self.alpha * self.y_train, self._K[:, i])
                grad = 1.0 - self.y_train[i] * f_i
                self.alpha[i] += self.learning_rate * grad
                # project to [0, C]
                if self.alpha[i] < 0.0:
                    self.alpha[i] = 0.0
                elif self.alpha[i] > self.C:
                    self.alpha[i] = self.C

        # Compute bias from support vectors with 0 < alpha < C
        sv = (self.alpha > 1e-6) & (self.alpha < self.C - 1e-6)
        if np.any(sv):
            f_sv = self._K[sv] @ (self.alpha * self.y_train)
            b_vals = self.y_train[sv] - f_sv
            self.b = b_vals.mean()
        else:
            self.b = 0.0

        return self

    def decision_function(self, X):
        K = self.kernel_function(self.X_train, X)
        scores = (self.alpha * self.y_train)[:, None] * K
        return scores.sum(axis=0) + self.b

    def predict(self, X):
        s = self.decision_function(X)
        y_pm1 = np.sign(s)
        y_pm1[y_pm1 == 0] = 1.0
        if np.array_equal(np.sort(self.classes_), np.array([-1, 1])):
            return y_pm1.astype(int)
        return np.where(y_pm1 == -1.0, self.classes_[0], self.classes_[1])


In [ ]:
def fit_ovr_kernel(X_tr, y_tr, **ksvm_params):
    classes = np.unique(y_tr)
    models = []
    for c in classes:
        y_bin = np.where(y_tr == c, 1, -1)
        m = KernelSVM(**ksvm_params)
        m.fit(X_tr, y_bin)
        models.append((c, m))
    return models

def predict_ovr_kernel(models, X):
    scores = np.column_stack([m.decision_function(X) for (_, m) in models])
    best = np.argmax(scores, axis=1)
    classes = np.array([c for (c, _) in models])
    return classes[best]

ovr_rbf = fit_ovr_kernel(X_tr, y_tr, C=2.0, gamma=0.2, epochs=150, learning_rate=0.01, random_state=0)
y_hat_rbf = predict_ovr_kernel(ovr_rbf, X_te)
acc_rbf = (y_hat_rbf == y_te).mean()

print(f"Wine accuracy (RBF kernel SVM, OVR): {acc_rbf:.3f}")
print(f"Wine accuracy (linear SVM, OVR):      {acc_lin:.3f}")


Wine accuracy (RBF kernel SVM, OVR): 1.000
Wine accuracy (linear SVM, OVR):      1.000
